# qrepro: reference reproductions

Every published-estimate number this repository claims, computed live and checked against the pinned literals in `references/values.py`. Each table renders the same objects the CLI prints and the tests assert; each cell asserts its result before printing it, so a dependency bump that moves a number fails this notebook.

| key | paper | scope |
|---|---|---|
| Beverland | [arXiv:2211.07629](https://arxiv.org/abs/2211.07629) | three application instances; Qualtran ships this model, so this checks wiring |
| GE19 | [arXiv:1905.09749v3](https://arxiv.org/abs/1905.09749) | logical reconciliation, the windowed construction, the physical rows |
| G2025 | [arXiv:2505.15917](https://arxiv.org/abs/2505.15917) | decomposed against GE19 through one model, not reproduced |

Sources, free parameters, conventions, tolerances and every named divergence: [`ASSUMPTIONS.md`](../ASSUMPTIONS.md).

Counting convention: the magic-state currency is `n_ccz = and_bloq + toffoli + cswap`; one Qualtran `And` = one `Toffoli` = one CCZ = 4 T = one GE19 "Toffoli" (sec. 3).

In [ ]:
import math
import re

from qualtran.bloqs.qft.approximate_qft import ApproximateQFT
from qualtran.bloqs.qft.qft_text_book import QFTTextBook
from qualtran.resource_counting import QECGatesCost, get_cost_value

from qrepro.algorithms import registry
from qrepro.algorithms.windowed_factoring import (
    make_ge19_windowed_modexp,
    windowed_term_breakdown,
)
from qrepro.references import (
    reproduce_2019_to_2025,
    reproduce_beverland,
    reproduce_ge19_logical,
    reproduce_ge19_physical,
    reproduce_ge19_windowed,
    windowed_total_ccz,
)

# Private import: the same normalisation reproduce_2019_to_2025 applies internally,
# so the counts shown in sec. 6 cannot drift from the rows computed from them.
from qrepro.references.decomposition import CONVENTIONS, _toffoli_counts
from qrepro.references.values import (
    BEVERLAND_TOL,
    G2025,
    G2025_TOL,
    GE19,
    GE19_TOL,
    GE19_WINDOWED,
    GE19_WINDOWED_ACHIEVED,
    GE19_WINDOWED_TOL,
    modexp_toffoli_coset,
    modexp_toffoli_reference,
    modexp_toffoli_windowed,
)
from qrepro.resource import rotation_synthesis_t_cost

N = GE19["n"]  # 2048
TABLE1 = GE19["table1_toffoli_billions"]
DEFAULT_WINDOW = (GE19_WINDOWED["exp_window"], GE19_WINDOWED["mul_window"])

In [ ]:
# Digits, optional thousands separators, optional exponent, optional unit suffix
# (12.3%, 5.64x, 17.97M). Anything else marks the column as text and left-aligns it.
_NUMERIC = re.compile(r"^[-+]?[\d,]*\.?\d+(?:[eE][-+]?\d+)?[%xM]?$")


def table(headers, rows):
    """Fixed-width table; columns whose cells all parse as numbers right-align."""
    cells = [[f"{v}" for v in row] for row in rows]
    widths = [max([len(h)] + [len(c[i]) for c in cells]) for i, h in enumerate(headers)]
    right = [
        all(_NUMERIC.match(c[i]) for c in cells if c[i]) for i in range(len(headers))
    ]

    def line(values):
        return "  ".join(
            f"{v:{'>' if r else '<'}{w}}" for v, r, w in zip(values, right, widths)
        ).rstrip()

    head = line(headers)
    print(head, "-" * len(head), sep="\n")
    for c in cells:
        print(line(c))


def check(name, ok, detail=""):
    """Assert an invariant and print it. Raises, so a bad cell stops the run."""
    if not ok:
        raise AssertionError(f"{name}{f' -- {detail}' if detail else ''}")
    print(f"  [ok] {name}{f'  ({detail})' if detail else ''}")


def within(value, target, rel):
    return abs(value - target) <= abs(target) * rel


def monotone(values, *, increasing, strict=False):
    """Monotonicity check. Direction is keyword-only so it cannot be defaulted."""
    pairs = list(zip(values, values[1:]))
    if increasing:
        return all(b > a if strict else b >= a for a, b in pairs)
    return all(b < a if strict else b <= a for a, b in pairs)

## 1. Primitives: rotation-synthesis sensitivity

Textbook and approximate QFT T-equivalent counts at n=32, swept over `rotation_synthesis_epsilon` and compared against Qualtran's own default.

The approximate QFT has zero arbitrary-angle rotations, so the ratio between the two measures synthesis cost alone. Rotation counts are eps-invariant, T-equivalent ratios are not, so neither ratio means anything without its eps. qrepro defaults to eps = 1e-10 via `T = 3*log2(1/eps)` (~100 T/rotation); Qualtran's default is ~11 T/rotation (sec. 2).

In [ ]:
qft, n_qft = registry["qft"], 32
EPSILONS = (1e-3, 1e-6, 1e-8, 1e-10, 1e-12, 1e-15)

rows = []
for eps in EPSILONS:
    tb = qft.logical_costs(
        qft.build_bloq(n=n_qft, variant="textbook"), rotation_synthesis_epsilon=eps
    )
    ap = qft.logical_costs(
        qft.build_bloq(n=n_qft, variant="approx"), rotation_synthesis_epsilon=eps
    )
    rows.append(
        dict(
            eps=eps,
            t_per_rot=rotation_synthesis_t_cost(eps),
            tb_direct=tb.t_count_direct,
            tb_ftqc=tb.t_count_ftqc,
            ap_ftqc=ap.t_count_ftqc,
            ratio=tb.t_count_ftqc / ap.t_count_ftqc,
            rotations=tb.rotation_count,
            ap_rotations=ap.rotation_count,
        )
    )

# Qualtran's own default precision, straight from the bloqs.
qtb = get_cost_value(QFTTextBook(bitsize=n_qft), QECGatesCost())
qap = get_cost_value(
    ApproximateQFT(bitsize=n_qft, phase_bitsize=n_qft // 2), QECGatesCost()
)


def t_equivalent(cost):
    counts = cost.total_t_and_ccz_count()
    return counts["n_t"] + 4 * counts["n_ccz"]


q_tb, q_ap = t_equivalent(qtb), t_equivalent(qap)
q_direct = int(qtb.t) + 4 * int(qtb.and_bloq)

table(
    ["epsilon", "T/rot", "textbook", "approx", "ratio"],
    [
        [
            "qualtran",
            f"{(q_tb - q_direct) / int(qtb.rotation):.1f}",
            f"{q_tb:,}",
            f"{q_ap:,}",
            f"{q_tb / q_ap:.3f}x",
        ]
    ]
    + [
        [
            f"{r['eps']:g}",
            r["t_per_rot"],
            f"{r['tb_ftqc']:,}",
            f"{r['ap_ftqc']:,}",
            f"{r['ratio']:.3f}x",
        ]
        for r in rows
    ],
)

default = next(r for r in rows if r["eps"] == 1e-10)
print(
    f"\nQFT({n_qft}) textbook rotations: {default['rotations']}"
    f"   direct T (eps-free): {default['tb_direct']:,}"
)
print(
    f"qrepro default eps=1e-10 -> {default['ratio']:.1f}x"
    f"   |   Qualtran default -> {q_tb / q_ap:.1f}x"
    f"   |   swept range {rows[0]['ratio']:.1f}x - {rows[-1]['ratio']:.1f}x"
)

check(
    "approximate QFT has no arbitrary-angle rotations",
    all(r["ap_rotations"] == 0 for r in rows),
)
check(
    "rotation count is eps-invariant",
    len({r["rotations"] for r in rows}) == 1,
    f"{default['rotations']} at every eps",
)
check("direct T-count is eps-invariant", len({r["tb_direct"] for r in rows}) == 1)
check(
    "ratio rises monotonically as eps tightens",
    monotone([r["ratio"] for r in rows], increasing=True, strict=True),
)
check(
    "Qualtran's looser default gives the smaller ratio", q_tb / q_ap < default["ratio"]
)

## 2. Beverland et al.

Qualtran's Beverland model evaluated on the paper's three application instances, against the targets in `values.py`. Qualtran ships this model, so agreement checks the wiring rather than showing independent convergence.

Targets are the paper's equations, not its Table I. Chemistry and factoring agree (sec. V-B L1398, sec. V-C L1422); quantum dynamics does not. Table I prints $C_{\min} = 1.5 \cdot 10^5$ and $R = 2.4 \cdot 10^6$, both irreconcilable with the paper's own (D3)/(D4): the measurement count alone ($M_{\text{Meas}} = 1.4 \cdot 10^6$) exceeds the printed step count 9.3x and the printed T-count is exactly 4x the formula's. (D3)/(D4) at the paper's $A = 0.53$, $B = 5.3$ give $1.4401 \cdot 10^6$ and $6.02 \cdot 10^5$, which is what Qualtran's model computes on the inputs as printed and what is targeted here (ASSUMPTIONS.md sec. 1). Code distances are the paper's printed values throughout, evaluated at the printed step count rather than the computed $C_{\min}$.

In [ ]:
beverland = reproduce_beverland()

table(
    ["instance", "metric", "qrepro", "target", "dev"],
    [
        [
            r.label,
            r.metric,
            f"{r.reproduced:.5g}",
            "" if r.target is None else f"{r.target:.5g}",
            "" if r.deviation is None else f"{r.deviation:+.2%}",
        ]
        for r in beverland.rows
    ],
)

worst = max(beverland.rows, key=lambda r: abs(r.deviation))
print(f"\nworst deviation: {worst.deviation:+.2%} on {worst.label}/{worst.metric}")

TOL_BY_METRIC = {
    "c_min": BEVERLAND_TOL["rel_c_min"],
    "t_states": BEVERLAND_TOL["rel_t_states"],
    "code_distance": 0.0,  # exact: the paper's own printed distances
}
for r in beverland.rows:
    tol = TOL_BY_METRIC[r.metric]
    check(
        f"{r.label}/{r.metric} within tolerance",
        abs(r.deviation) <= tol,
        f"{r.deviation:+.2%} <= {tol:.0%}",
    )

## 3. GE19 logical

GE19's closed form and Qualtran's `ModExp` call-graph count, side by side with the three modexp regimes GE19 prices. The two differ by ~64x: `ModExp` implements GE19's reference (non-windowed) construction at `20*n_e*n^2`, not the paper's optimized result.

`QubitCount`, `AlgorithmSummary.from_bloq` and `decompose_bloq` are never called on `ModExp`, since they trace wires in O(gates) and hang at n >= 128, so the logical-qubit count comes from GE19's `3n + 0.002*n*lg n`.

In [ ]:
logical = reproduce_ge19_logical()
ne_shor, ne_eh = 2 * N, 1.5 * N

table(
    ["quantity", "value", "target", "dev"],
    [
        [
            "logical qubits (formula)",
            f"{logical.logical_qubits_formula:.1f}",
            GE19["logical_qubits"],
            "",
        ],
        [
            "Toffoli (GE19 formula)",
            f"{logical.toffoli_formula:.4e}",
            f"{GE19['toffoli_count']:.1e}",
            f"{logical.toffoli_formula / GE19['toffoli_count'] - 1:+.2%}",
        ],
        ["Qualtran ModExp (n_ccz)", f"{logical.modexp_ccz_count:,}", "", ""],
        ["ModExp / GE19 formula", f"{logical.divergence_ratio:.2f}x", "", ""],
    ],
)

table(
    ["modexp regime", "source", "n_e", "Toffoli"],
    [
        [
            "reference  20*ne*n^2",
            "S2.2 L522",
            "2n",
            f"{modexp_toffoli_reference(N, ne_shor):.4e}",
        ],
        [
            "coset       8*ne*n^2",
            "S2.4 L547",
            "2n",
            f"{modexp_toffoli_coset(N, ne_shor):.4e}",
        ],
        [
            "windowed 24*ne*n^2/lg^2n",
            "S2.5 L602",
            "1.5n",
            f"{modexp_toffoli_windowed(N, ne_eh):.4e}",
        ],
        ["optimized (Table 1)", "Table 1", "1.5n", f"{GE19['toffoli_count']:.4e}"],
        [
            "MEASURED Qualtran ModExp",
            "call graph",
            "2n",
            f"{logical.modexp_ccz_count:.4e}",
        ],
    ],
)
print(
    f"\nreference -> windowed: coset 2.5x * windowing {math.log2(N) ** 2 / 3:.1f}x"
    f" (lg^2n/3) * exponent 1.33x = {2.5 * math.log2(N) ** 2 / 3 * 4 / 3:.0f}x"
)

check(
    "ModExp/formula divergence in band",
    GE19_TOL["divergence_lo"] <= logical.divergence_ratio <= GE19_TOL["divergence_hi"],
    f"{logical.divergence_ratio:.2f}x",
)
check(
    "logical-qubit formula rounds to the abstract's value",
    round(logical.logical_qubits_formula) == GE19["logical_qubits"],
)

`n_ccz/(n_e*n^2)` measured across a 64-fold range in n. A constant identifies the non-windowed reference regime; a windowed construction falls like `1/lg^2 n`, which section 4 measures.

The constant is half the 20 GE19 derives (sec. 2.2 L522) for the construction Qualtran documents. The factor of two is the adder primitive: GE19 prices Cuccaro's adder at `2n` Toffolis, Qualtran's `Add` is Gidney's temporary-AND adder at `n-1` ANDs with the carry uncomputed by measurement. The closed form printed below is a regression pin, not evidence, because its coefficient was fitted to the measurement. The evidence is the scaling.

In [ ]:
sizes, coeffs = zip(*logical.coefficient_series)
table(["n", "n_ccz/(ne*n^2)"], [[n, f"{c:.5f}"] for n, c in logical.coefficient_series])

closed_form = 10 * ne_shor * N**2 + 5 * ne_shor * N
print(f"\nconverges to {coeffs[-1]:.5f} -- constant, no 1/lg^2(n) factor")
print(
    f"spread over n in [{sizes[0]}, {sizes[-1]}]: "
    f"{(coeffs[0] - coeffs[-1]) / coeffs[-1]:.2%}"
)
print(
    f"that constant is {logical.measured_coefficient:.3f} = half the "
    f"{logical.ge19_reference_coefficient:.0f} of S2.2 L522 (adder primitive)"
)
print(f"closed form 10*ne*n^2 + 5*ne*n = {closed_form:,}  [regression pin, fitted]")

check(
    "coefficient series is monotone non-increasing",
    monotone(list(coeffs), increasing=False),
)
check(
    "coefficient converges to 10", within(coeffs[-1], 10.0, 0.01), f"{coeffs[-1]:.5f}"
)
check("series is flat, not falling", (coeffs[0] - coeffs[-1]) / coeffs[-1] < 0.02)
check(
    "closed form is exact against the call graph",
    closed_form == logical.modexp_ccz_count,
)

## 4. GE19 windowed construction

`algorithms/windowed_factoring.py` builds what GE19 costs, windowed exponentiation over windowed multiplication over the coset representation (sec. 2.4-2.5) at Ekera-Hastad `n_e = 1.5n`, from stock Qualtran leaves:

```
WindowedModExp             ceil(n_e/w_e) uncontrolled multiplications  (L590)
  WindowedModMul           2 multiply-add passes                       (anc:171, L694)
    WindowedMultiplyAdd    ceil((n+g_pad+2)/w_m) lookup additions      (L590)
      LookupAddition
        QROAMClean(log_block_sizes=(0,))   lookup      (L594)
        Add(QUInt(width))                  addition    (L593)
        QROAMClean(...).adjoint()          unlookup    (L595)
```

This is a second derivation of the 2.7e9 regime that does not go through the paper's closed forms. Every window and padding parameter is GE19-published (`g_exp = g_mul = 5`, `g_pad = 2 lg n + lg n_e + 10`, L690).

"Bridged" doubles the adder term and only the adder term, converting Qualtran's Gidney AND-adder to GE19's Cuccaro convention. It is reported beside the unbridged figure, which is the number this pipeline computes.

In [ ]:
windowed = reproduce_ge19_windowed()

table(
    ["n", "n_e", "g_pad", "window", "total n_ccz", "adder", "lookup", "unlookup"],
    [
        [
            i.n,
            i.exp_bitsize,
            i.coset_padding,
            f"({i.exp_window},{i.mul_window})",
            f"{i.total_ccz:,}",
            f"{i.adder_ccz / i.total_ccz:.2%}",
            f"{i.lookup_ccz / i.total_ccz:.2%}",
            f"{i.unlookup_ccz / i.total_ccz:.2%}",
        ]
        for i in windowed.instances
    ],
)
table(
    ["n", "measured", "/T1", "bridged", "bridged/T1", "/24 form", "/16 form"],
    [
        [
            i.n,
            f"{i.total_ccz:.4e}",
            f"{i.table1_ratio:.4f}",
            f"{i.bridged_ccz:.4e}",
            f"{i.bridged_table1_ratio:.4f}",
            f"{i.total_ccz / i.closed_form_24:.4f}",
            f"{i.total_ccz / i.closed_form_16:.4f}",
        ]
        for i in windowed.instances
    ],
)
print("\n16*ne*n^2/lg^2n is L602's 24 in Qualtran's adder currency, not a published")
print("constant: the 24 = 2x4x3 has its 3 = 2 adder + 1 lookup become 2 = 1 + 1.")

for i in windowed.instances:
    key, pin = f"n{i.n}", GE19_WINDOWED_ACHIEVED[f"n{i.n}"]
    check(
        f"n={i.n} count pinned exactly",
        (i.total_ccz, i.adder_ccz, i.lookup_ccz, i.unlookup_ccz, i.bridged_ccz)
        == (
            pin["total_ccz"],
            pin["adder_ccz"],
            pin["lookup_ccz"],
            pin["unlookup_ccz"],
            pin["bridged_ccz"],
        ),
    )
    check(
        f"n={i.n} terms sum to total",
        i.adder_ccz + i.lookup_ccz + i.unlookup_ccz == i.total_ccz,
    )
    check(
        f"n={i.n} vs Table 1 in declared band",
        GE19_WINDOWED_TOL["table1_lo"][key]
        <= i.table1_ratio
        <= GE19_WINDOWED_TOL["table1_hi"][key],
        f"{i.table1_ratio:.4f}",
    )
    check(
        f"n={i.n} bridged vs Table 1 in declared band",
        GE19_WINDOWED_TOL["bridged_table1_lo"][key]
        <= i.bridged_table1_ratio
        <= GE19_WINDOWED_TOL["bridged_table1_hi"][key],
        f"{i.bridged_table1_ratio:.4f}",
    )
    check(
        f"n={i.n} vs 16*ne*n^2/lg^2n",
        within(i.total_ccz, i.closed_form_16, GE19_WINDOWED_TOL["rel_closed_form_16"]),
    )

### Regime identification

`total/(n_e*n^2)` over n in {128 ... 8192}, which must fall like `1/lg^2 n`. Section 3 ran the same measurement on `ModExp` and got a constant. The test uses no external number: it separates the two regimes on scaling alone.

In [ ]:
ns, cs = zip(*windowed.coefficient_series)
table(
    ["n", "coefficient", "x lg^2n"],
    [
        [n, f"{c:.6f}", f"{c * math.log2(n) ** 2:.4f}"]
        for n, c in windowed.coefficient_series
    ],
)

fall = cs[0] / cs[-1]
lg_rise = math.log2(ns[-1]) ** 2 / math.log2(ns[0]) ** 2
scaled = dict(windowed.coefficient_series)[N] * math.log2(N) ** 2
print(
    f"\ncoefficient falls {fall:.2f}x while lg^2n rises {lg_rise:.2f}x"
    f" ({math.log2(ns[0]) ** 2:.0f} -> {math.log2(ns[-1]) ** 2:.0f})"
)
print(f"residual drift in the product: {fall / lg_rise:.2f}x")

check(
    "coefficient falls strictly at every step",
    monotone(list(cs), increasing=False, strict=True),
)
check(
    "1/lg^2n accounts for most of the fall",
    lg_rise / fall > 0.5,
    f"{lg_rise:.2f}x of {fall:.2f}x",
)
check(
    f"coeff(n={N}) * lg^2n in declared band",
    GE19_WINDOWED_TOL["falloff_lg2_lo"]
    <= scaled
    <= GE19_WINDOWED_TOL["falloff_lg2_hi"],
    f"{scaled:.4f}",
)
# Same measurement, both regimes.
modexp_spread = coeffs[0] / coeffs[-1]
check(
    "windowed falls where the reference regime is flat",
    fall > 2 and modexp_spread < 1.02,
    f"windowed {fall:.2f}x vs ModExp {modexp_spread:.3f}x",
)

### Window grid

The full `(w_e, w_m)` grid at n=2048, then the per-n cost argmin beside GE19's published `(5, 5)` at each tabulated size.

`(5, 5)` comes from GE19 L690; the grid minimum checks that value rather than justifying it. The per-n window is the cost argmin: at n=1024 that lands further from Table 1 than `(5, 5)` would, so both are printed.

In [ ]:
grid = sorted(windowed.window_grid, key=lambda r: r[2])
table(
    ["w_e", "w_m", "k", "2^k", "total n_ccz", "/T1"],
    [
        [
            we,
            wm,
            we + wm,
            f"{2 ** (we + wm):,}",
            f"{t:,}",
            f"{t / (TABLE1['n2048'] * 1e9):.4f}",
        ]
        for we, wm, t in grid
    ],
)
print(
    f"\n{len(grid)} window pairs at n={N}; span "
    f"{grid[0][2] / 1e9:.3f}e9 - {grid[-1][2] / 1e9:.3f}e9 CCZ"
    f" ({grid[-1][2] / grid[0][2]:.1f}x)"
)

rows = []
for i in windowed.instances:
    default_ccz = windowed_total_ccz(i.n, *DEFAULT_WINDOW)
    rows.append(
        [
            i.n,
            f"({i.exp_window},{i.mul_window})",
            f"{i.total_ccz:,}",
            f"{i.table1_ratio:.4f}",
            f"{default_ccz:,}",
            f"{default_ccz / i.table1_toffoli:.4f}",
            "further" if i.table1_ratio < default_ccz / i.table1_toffoli else "=",
        ]
    )
table(
    [
        "n",
        "argmin",
        "argmin n_ccz",
        "argmin/T1",
        "(5,5) n_ccz",
        "(5,5)/T1",
        "argmin vs T1",
    ],
    rows,
)

check(
    "grid minimum is GE19's published default",
    windowed.window_argmin == DEFAULT_WINDOW,
    f"{windowed.window_argmin}",
)
check(
    "cost argmin is never the window closest to Table 1",
    all(
        i.table1_ratio <= windowed_total_ccz(i.n, *DEFAULT_WINDOW) / i.table1_toffoli
        for i in windowed.instances
    ),
)
check("window choice matters (grid is not flat)", grid[-1][2] / grid[0][2] > 5)

### Structural guards

Three properties the count depends on: the gate-field split (`and_bloq` alone drops the entire unlookup term), the absence of a CSwap (GE19's windowed multiplications are uncontrolled, so the register relabel is bookkeeping), and the collapsed call graph (`QROAMClean` built from bitsize rather than table data, since from data the graph never collapses and the count never terminates). Then the runway-exclusion uplift.

In [ ]:
bloq = make_ge19_windowed_modexp(N)
gates = get_cost_value(bloq, QECGatesCost())
total = int(gates.total_t_and_ccz_count(ts_per_rotation=0)["n_ccz"])
nodes = bloq.call_graph()[0].number_of_nodes()

off = windowed_total_ccz(N, *DEFAULT_WINDOW)
on = windowed_term_breakdown(
    make_ge19_windowed_modexp(N, runway_sep=GE19_WINDOWED["ge19_runway_sep"])
).total_ccz

table(
    ["property", "value", "note"],
    [
        ["and_bloq (lookup + adder)", f"{int(gates.and_bloq):,}", "QECGatesCost field"],
        ["toffoli (unlookup)", f"{int(gates.toffoli):,}", "the whole unlookup term"],
        ["cswap", f"{int(gates.cswap):,}", "uncontrolled, so none emitted"],
        ["n_ccz total", f"{total:,}", "authoritative currency"],
        [
            "and_bloq-only error",
            f"{(total - int(gates.and_bloq)) / total:.2%}",
            "dropped by an and_bloq-only extraction",
        ],
        ["call-graph nodes", nodes, "O(10), not O(n_e)"],
        [
            f"runways at g_sep={GE19_WINDOWED['ge19_runway_sep']}",
            f"{(on - off) / off:+.2%}",
            "excluded from the count by default",
        ],
        [
            "vs stock ModExp",
            f"{logical.modexp_ccz_count / off:.1f}x",
            "same currency, same pinned qualtran",
        ],
    ],
)

check("no CSwap is emitted", int(gates.cswap) == 0)
check("cost spans both gate fields", int(gates.and_bloq) > 0 and int(gates.toffoli) > 0)
check("and_bloq + toffoli == n_ccz", int(gates.and_bloq) + int(gates.toffoli) == total)
check("call graph stays collapsed", nodes < 30, f"{nodes} nodes")
check(
    "runway uplift matches the pin",
    within((on - off) / off, GE19_WINDOWED_ACHIEVED["runway_uplift_n2048"], 0.1),
    f"{(on - off) / off:+.2%}",
)
check("windowed is ~100x below stock ModExp", off * 50 < logical.modexp_ccz_count)

## 5. GE19 physical

GE19's formula count, not the `ModExp` count, through the CCZ2T grid search at `phys_err = 1e-3`, `cycle_time_us = 1.0`, `n_algo_qubits = 6189`. Both rows use the same search and differ only in `n_factories`.

Both free parameters are GE19-published: `error_budget = 0.31` is the paper's retry risk (Table 3; L1086 defines it as "the overall probability of errors occurring", which is what Qualtran's `error_budget` means, so no proxy is needed), and `n_factories = 28` is Table 2's factory count for the parallel row.

Conventions do not mix. qrepro emits a per-run duration and has no retry model; GE19 Table 2 quotes expected, Table 3 quotes per run, and L1096 gives the conversion $t/(1-\epsilon)$. Every row below is labelled with the comparison it makes (sec. 3).

In [ ]:
physical = reproduce_ge19_physical()
t3 = GE19["physical_rows"]["table3_authoritative"]

table(
    ["row", "metric", "qrepro", "GE19", "dev"],
    [
        [
            r.label,
            r.metric,
            f"{r.reproduced:.5g}",
            "" if r.target is None else f"{r.target:.4g}",
            "" if r.deviation is None else f"{r.deviation:+.1%}",
        ]
        for r in physical.rows
    ],
)

devs = [abs(r.deviation) for r in physical.rows if r.deviation is not None]
print(f"\nall {len(devs)} deviations within {min(devs):.1%} - {max(devs):.1%}")
print(
    f"factory search selects d1={physical.parallel.factory_l1_d}, "
    f"d2={physical.parallel.factory_l2_d}; GE19 Table 3 publishes "
    f"d1={t3['d1']}, d2={t3['d2']}"
)
print(
    f"1-factory data code distance d={physical.one_factory.code_distance}, "
    f"budget_satisfied={physical.one_factory.budget_satisfied}"
)

for r in physical.rows:
    if r.deviation is None:
        continue
    tol = GE19_TOL["rel_qubits" if "qubits" in r.metric else "rel_runtime"]
    check(
        f"{r.label}/{r.metric} within tolerance",
        abs(r.deviation) <= tol,
        f"{r.deviation:+.1%} <= {tol:.0%}",
    )
check(
    "grid search finds GE19's own factory",
    (physical.parallel.factory_l1_d, physical.parallel.factory_l2_d)
    == (t3["d1"], t3["d2"]),
)
check(
    "GE19's Tables 2 and 3 reconcile via the published retry risk",
    within(
        t3["runtime_hr_per_run"] / (1 - t3["retry"]) / 24,
        GE19["physical_rows"]["parallel"]["runtime_days"],
        0.01,
    ),
)

Sensitivity sweeps over `error_budget` and `n_factories`, showing how far the reproduction moves when either published value is varied.

The 1-factory runtime comparison cannot be aligned: GE19 publishes only an expected runtime for that scenario and Table 3's per-run figure covers the n=2048 optimum only, so there is no per-run target for that layout. Both readings are printed; resolving it needs a retry risk GE19 does not publish.

In [ ]:
table(
    ["eb", "1f qubits", "1f hr/run", "1f d", "28f qubits", "28f hr/run", "28f d"],
    [
        [
            f"{eb:.2f}",
            f"{o.physical_qubits / 1e6:.2f}M",
            f"{o.wall_time_us / 3.6e9:.2f}",
            o.code_distance,
            f"{p.physical_qubits / 1e6:.2f}M",
            f"{p.wall_time_us / 3.6e9:.2f}",
            p.code_distance,
        ]
        for eb, o, p in physical.sweep
    ],
)
table(
    ["nf", "qubits", "hr/run", "d_data", "d1", "d2", "fail"],
    [
        [
            nf,
            f"{c.physical_qubits / 1e6:.2f}M",
            f"{c.wall_time_us / 3.6e9:.2f}",
            c.code_distance,
            c.factory_l1_d,
            c.factory_l2_d,
            f"{c.failure_prob:.3f}",
        ]
        for nf, c in physical.factory_sweep
    ],
)

retry = t3["retry"]
per_run = physical.one_factory_runtime_hr
target = physical.one_factory_target_runtime_hr_expected
print(f"\n1-factory runtime vs GE19's expected {target:.0f} hr:")
print(
    f"  per run   {per_run:.3f} hr  ({per_run / target - 1:+.1%})   <- the table above"
)
print(
    f"  expected  {per_run / (1 - retry):.3f} hr  "
    f"({per_run / (1 - retry) / target - 1:+.1%})   <- same number, other convention"
)

par = [p for _, _, p in physical.sweep]
check(
    "parallel row is insensitive to the error budget",
    max(p.physical_qubits for p in par) / min(p.physical_qubits for p in par) < 1.3,
)
check(
    "every swept configuration satisfies its budget",
    all(c.budget_satisfied for _, c in physical.factory_sweep),
)
check(
    "the two conventions bracket the target",
    (per_run < target) != (per_run / (1 - retry) < target),
)

## 6. 2019 to 2025 decomposition

G2025's logical counts (Table 5: 6.5e9 Toffolis on 1399 logical qubits at n=2048) through the same 2019-era CCZ2T model as GE19 at the same factory count, so no layout difference enters the algorithmic share.

The Toffoli conventions differ and are normalised first. GE19 Table 1 is per run (L1788: "does not account for the chance of retrying"); G2025 Table 5 is expected per factoring, already aggregating $E(\text{shots}) = 9.2$. Physical qubits are a per-shot resource, so feeding both in raw compares different quantities. Both normalisations are computed.

This is a decomposition, not a reproduction: G2025's published < 1e6 rests on yoked surface codes and magic-state cultivation, which have no representation in any open cost model checked (Qualtran, Azure QRE, pyLIQTR). The residual gap is reported, not closed.

In [ ]:
print(
    f"G2025 Table 5: {G2025['toffoli_count']:.1e} Toffoli "
    f"(expected/factoring, E(shots)={G2025['expected_shots']}), "
    f"{G2025['logical_qubits']} logical qubits vs GE19's {GE19['logical_qubits']}"
)

ratios, ge19_qubits, g2025_qubits = [], [], []
for convention in CONVENTIONS:
    decomp = reproduce_2019_to_2025(convention)
    ge19_t, g2025_t = _toffoli_counts(convention)
    print(
        f"\nconvention={convention}  (GE19 {ge19_t:.4e} Toffoli, G2025 {g2025_t:.4e})"
    )
    table(
        ["factories", "GE19 qubits", "G2025 qubits", "algorithmic x"],
        [
            [
                r.n_factories,
                f"{r.ge19.physical_qubits / 1e6:.2f}M",
                f"{r.g2025.physical_qubits / 1e6:.2f}M",
                f"{r.algorithmic_ratio:.2f}x",
            ]
            for r in decomp.factory_rows
        ],
    )
    ratios += [r.algorithmic_ratio for r in decomp.factory_rows]
    ge19_qubits += [r.ge19.physical_qubits / 1e6 for r in decomp.factory_rows]
    g2025_qubits += [r.g2025.physical_qubits / 1e6 for r in decomp.factory_rows]

spread_conv = max(ratios[i] - ratios[i + 3] for i in range(3))
spread_nf = max(max(ratios[:3]) - min(ratios[:3]), max(ratios[3:]) - min(ratios[3:]))
print(
    f"\nrange across both conventions and all factory counts: "
    f"{min(ratios):.2f}x - {max(ratios):.2f}x"
)
print(
    f"  factory count spreads it by up to {spread_nf:.2f}; "
    f"convention by up to {spread_conv:.2f}"
)
print(
    f"model puts GE19 at {min(ge19_qubits):.2f}-{max(ge19_qubits):.2f} M against its "
    f"published {GE19['physical_rows']['parallel']['qubits_M']} M, i.e. "
    f"{1 - max(ge19_qubits) / 20:.0%}-{1 - min(ge19_qubits) / 20:.0%} low"
)
print(
    f"G2025 published target: {G2025['published_physical_qubits']} physical qubits -- "
    f"this model floors out in the millions"
)

check(
    "ratio range within the declared band",
    G2025_TOL["algo_ratio_lo"] <= min(ratios)
    and max(ratios) <= G2025_TOL["algo_ratio_hi"],
    f"{min(ratios):.2f}x - {max(ratios):.2f}x",
)
check(
    "factory count spreads the range more than the convention does",
    spread_nf > spread_conv,
    f"{spread_nf:.2f} vs {spread_conv:.2f}",
)
check(
    "model does not reach G2025's sub-million target",
    min(g2025_qubits) > 1.0,
    f"floor {min(g2025_qubits):.2f} M",
)